# Notebook 04 — Preprocessing Pipeline and Train/Test Split

**Project:** Machine Learning-Based Prediction of Parkinson’s Disease Progression Using PPMI  
**Stage:** Preprocessing only — no machine learning model training  
**Primary outcome:** Rapid motor progression at V06, based on annualized change in MDS-UPDRS Part III (`NP3TOT`)  
**Input from Notebook 03:** `08_feature_matrix_predictors_missingness_le_30pct.csv`

---

## Objective

This notebook creates a reproducible, leakage-safe preprocessing pipeline for the baseline predictor matrix created in Notebook 03. It performs:

1. Loading and verifying the Notebook 03 feature matrix.
2. Defining predictors (`X`) and target (`y`) without leakage.
3. Removing constant predictors using the training data only.
4. Creating a stratified train/test split.
5. Fitting imputation and encoding/scaling steps on the training set only.
6. Transforming both training and test sets using the same fitted preprocessing pipeline.
7. Saving the processed matrices and quality-control outputs.

**No machine learning model is trained in this notebook.**

## Scientific Background

For a longitudinal prediction project, predictors must be restricted to information available at baseline or before the prediction time point. In this project, the primary target is rapid motor progression by V06, defined in Notebook 02 from follow-up `NP3TOT`. Therefore, columns derived from follow-up outcomes, such as `delta_NP3TOT` and `annualized_delta_NP3TOT`, must not be included as predictors.

This notebook also separates train and test sets before fitting preprocessing steps. This avoids information from the test set influencing imputation, scaling, encoding, or feature-selection decisions.

## Dataset Verification

Expected input file:

```text
MyDrive/PPMI_PD_Progression/outputs/notebook_03_baseline_predictors/08_feature_matrix_predictors_missingness_le_30pct.csv
```

Expected core columns:

```text
PATNO
rapid_progression_q75
delta_NP3TOT
annualized_delta_NP3TOT
```

Only `rapid_progression_q75` will be used as the primary target in this notebook. The continuous outcome columns are retained for audit only and excluded from predictors.

In [ ]:
# ============================================================
# 00. Environment setup
# ============================================================

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold

import joblib

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TEST_SIZE = 0.20

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

print("Environment ready.")
print("pandas:", pd.__version__)

In [ ]:
# ============================================================
# 01. Mount Google Drive
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception as e:
    IN_COLAB = False
    print("Not running in Google Colab, or Google Drive mount failed.")
    print("Error:", e)

In [ ]:
# ============================================================
# 02. Define project paths
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/PPMI_PD_Progression")

INPUT_DIR = PROJECT_DIR / "outputs" / "notebook_03_baseline_predictors"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "notebook_04_preprocessing"

INPUT_FILE = INPUT_DIR / "08_feature_matrix_predictors_missingness_le_30pct.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("INPUT_FILE exists:", INPUT_FILE.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}\n"
        "Please run Notebook 03 first and confirm that the output file exists."
    )

In [ ]:
# ============================================================
# 03. Load Notebook 03 feature matrix
# ============================================================

df = pd.read_csv(INPUT_FILE)

print("Shape:", df.shape)
display(df.head())

required_cols = ["PATNO", "rapid_progression_q75", "delta_NP3TOT", "annualized_delta_NP3TOT"]
missing_required = [c for c in required_cols if c not in df.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

if df["PATNO"].duplicated().any():
    dup_n = int(df["PATNO"].duplicated().sum())
    raise ValueError(f"Duplicate PATNO values found: {dup_n}")

if df["rapid_progression_q75"].isna().any():
    raise ValueError("Target column rapid_progression_q75 contains missing values.")

print("Required columns verified.")
print("Unique participants:", df["PATNO"].nunique())
print("Target distribution:")
display(df["rapid_progression_q75"].value_counts(dropna=False).rename("n").to_frame())

In [ ]:
# ============================================================
# 04. Define leakage-safe predictors and target
# ============================================================

id_col = "PATNO"
target_col = "rapid_progression_q75"

leakage_or_audit_cols = [
    "rapid_progression_q75",
    "delta_NP3TOT",
    "annualized_delta_NP3TOT",
]

feature_cols = [c for c in df.columns if c not in ([id_col] + leakage_or_audit_cols)]

X = df[feature_cols].copy()
y = df[target_col].astype(int).copy()
ids = df[id_col].copy()

print("Candidate predictor count:", len(feature_cols))
print("Candidate predictors:")
for c in feature_cols:
    print("-", c)

leakage_check = [c for c in X.columns if c.startswith("followup_") or "delta" in c.lower() or c == target_col]
print("\nLeakage-like columns found in X:", leakage_check)
if leakage_check:
    raise ValueError(f"Potential leakage columns found in predictors: {leakage_check}")

In [ ]:
# ============================================================
# 05. Stratified train/test split
# ============================================================

X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X,
    y,
    ids,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

split_summary = pd.DataFrame({
    "set": ["train", "test", "overall"],
    "n": [len(y_train), len(y_test), len(y)],
    "positive_n": [int(y_train.sum()), int(y_test.sum()), int(y.sum())],
    "negative_n": [int((1-y_train).sum()), int((1-y_test).sum()), int((1-y).sum())],
})
split_summary["positive_percent"] = 100 * split_summary["positive_n"] / split_summary["n"]

display(split_summary)
split_summary.to_csv(OUTPUT_DIR / "01_train_test_split_summary.csv", index=False)

In [ ]:
# ============================================================
# 06. Remove constant predictors using training data only
# ============================================================

train_unique_counts = X_train.nunique(dropna=True)
constant_cols = train_unique_counts[train_unique_counts <= 1].index.tolist()

print("Constant predictors identified from training data:", constant_cols)

X_train_reduced = X_train.drop(columns=constant_cols)
X_test_reduced = X_test.drop(columns=constant_cols)

constant_report = pd.DataFrame({
    "dropped_constant_predictor": constant_cols
})
constant_report.to_csv(OUTPUT_DIR / "02_dropped_constant_predictors.csv", index=False)

print("Predictors before dropping constants:", X_train.shape[1])
print("Predictors after dropping constants:", X_train_reduced.shape[1])

In [ ]:
# ============================================================
# 07. Define variable types
# ============================================================

# These definitions are intentionally explicit and transparent.
continuous_features = [
    "ENROLL_AGE",
    "baseline_NP3TOT",
    "part1p_NP1PTOT",
    "derived_years_since_PD_diagnosis",
    "part2_NP2PTOT",
    "vitals_SYSSUP",
    "vitals_DIASUP",
    "vitals_HRSUP",
    "part1_NP1RTOT",
    "moca_MCATOT",
    "derived_years_since_symptom_onset",
    "vitals_WGTKG",
    "vitals_HTCM",
    "derived_BMI",
]

binary_features = [
    "ENRLLRRK2",
    "ENRLGBA",
    "ENRLSNCA",
    "ENRLPRKN",
]

categorical_features = [
    "baseline_NHY",
    "pddx_DXBRADY",
    "primdiag_PRIMDIAG",
    "pddx_DXRIGID",
    "pddx_DXTREMOR",
    "pddx_DOMSIDE",
    "pddx_DXPOSINS",
]

# Keep only features that survived the constant-column filter.
available_cols = set(X_train_reduced.columns)

continuous_features = [c for c in continuous_features if c in available_cols]
binary_features = [c for c in binary_features if c in available_cols]
categorical_features = [c for c in categorical_features if c in available_cols]

assigned = set(continuous_features + binary_features + categorical_features)
unassigned = [c for c in X_train_reduced.columns if c not in assigned]

if unassigned:
    print("Unassigned columns detected. They will be treated as continuous by default:")
    for c in unassigned:
        print("-", c)
    continuous_features += unassigned

feature_type_table = pd.DataFrame({
    "predictor": continuous_features + binary_features + categorical_features,
    "feature_type": (
        ["continuous"] * len(continuous_features)
        + ["binary"] * len(binary_features)
        + ["categorical"] * len(categorical_features)
    )
})

display(feature_type_table)
feature_type_table.to_csv(OUTPUT_DIR / "03_feature_type_dictionary.csv", index=False)

print("Continuous:", len(continuous_features))
print("Binary:", len(binary_features))
print("Categorical:", len(categorical_features))

In [ ]:
# ============================================================
# 08. Build preprocessing pipeline
# ============================================================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("continuous", numeric_transformer, continuous_features),
        ("binary", binary_transformer, binary_features),
        ("categorical", categorical_transformer, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

print("Preprocessing pipeline created.")
print(preprocessor)

In [ ]:
# ============================================================
# 09. Fit preprocessing on training data only and transform train/test
# ============================================================

X_train_processed = preprocessor.fit_transform(X_train_reduced)
X_test_processed = preprocessor.transform(X_test_reduced)

feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train_reduced.index)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test_reduced.index)

print("Processed train shape:", X_train_processed_df.shape)
print("Processed test shape:", X_test_processed_df.shape)

print("Missing values in processed train:", int(X_train_processed_df.isna().sum().sum()))
print("Missing values in processed test:", int(X_test_processed_df.isna().sum().sum()))

display(X_train_processed_df.head())

In [ ]:
# ============================================================
# 10. Save train/test processed data and metadata
# ============================================================

# Raw split files before preprocessing
train_raw = pd.concat([
    ids_train.reset_index(drop=True).rename("PATNO"),
    y_train.reset_index(drop=True).rename(target_col),
    X_train_reduced.reset_index(drop=True)
], axis=1)

test_raw = pd.concat([
    ids_test.reset_index(drop=True).rename("PATNO"),
    y_test.reset_index(drop=True).rename(target_col),
    X_test_reduced.reset_index(drop=True)
], axis=1)

# Processed split files
train_processed = pd.concat([
    ids_train.reset_index(drop=True).rename("PATNO"),
    y_train.reset_index(drop=True).rename(target_col),
    X_train_processed_df.reset_index(drop=True)
], axis=1)

test_processed = pd.concat([
    ids_test.reset_index(drop=True).rename("PATNO"),
    y_test.reset_index(drop=True).rename(target_col),
    X_test_processed_df.reset_index(drop=True)
], axis=1)

train_raw.to_csv(OUTPUT_DIR / "04_train_raw_split_before_preprocessing.csv", index=False)
test_raw.to_csv(OUTPUT_DIR / "05_test_raw_split_before_preprocessing.csv", index=False)

train_processed.to_csv(OUTPUT_DIR / "06_train_processed_matrix.csv", index=False)
test_processed.to_csv(OUTPUT_DIR / "07_test_processed_matrix.csv", index=False)

pd.DataFrame({"processed_feature_name": feature_names}).to_csv(
    OUTPUT_DIR / "08_processed_feature_names.csv", index=False
)

joblib.dump(preprocessor, OUTPUT_DIR / "09_fitted_preprocessing_pipeline.joblib")

print("Saved outputs to:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 11. Sensitivity feature set without baseline NP3TOT
# ============================================================

# Baseline motor severity is clinically important and allowed as a baseline predictor.
# However, because the outcome is change in NP3TOT, later modeling should include a sensitivity analysis
# excluding baseline_NP3TOT to evaluate dependence on baseline severity/regression-to-the-mean effects.

sensitivity_exclude = ["baseline_NP3TOT"]

sensitivity_feature_names = [
    c for c in feature_names
    if not (
        c == "continuous__baseline_NP3TOT"
        or c.endswith("__baseline_NP3TOT")
        or "baseline_NP3TOT" in c
    )
]

sensitivity_report = pd.DataFrame({
    "sensitivity_analysis": ["exclude_baseline_NP3TOT"],
    "reason": [
        "Outcome is based on change in NP3TOT; exclude baseline_NP3TOT in sensitivity analysis to assess robustness."
    ],
    "processed_feature_count_primary": [len(feature_names)],
    "processed_feature_count_sensitivity": [len(sensitivity_feature_names)]
})

display(sensitivity_report)
sensitivity_report.to_csv(OUTPUT_DIR / "10_sensitivity_feature_set_plan.csv", index=False)
pd.DataFrame({"processed_feature_name": sensitivity_feature_names}).to_csv(
    OUTPUT_DIR / "11_processed_feature_names_without_baseline_NP3TOT.csv",
    index=False
)

In [ ]:
# ============================================================
# 12. Quality Control Checklist
# ============================================================

qc_items = []

def add_qc(item, status, detail):
    qc_items.append({"qc_item": item, "status": status, "detail": detail})

add_qc(
    "Notebook 03 feature matrix loaded",
    "PASS" if df.shape[0] > 0 else "FAIL",
    f"Rows: {df.shape[0]}; columns: {df.shape[1]}"
)

add_qc(
    "Unique participant IDs",
    "PASS" if not df["PATNO"].duplicated().any() else "FAIL",
    f"Duplicate PATNO count: {int(df['PATNO'].duplicated().sum())}"
)

add_qc(
    "Target available and non-missing",
    "PASS" if df[target_col].notna().all() else "FAIL",
    f"Non-missing target values: {int(df[target_col].notna().sum())}"
)

add_qc(
    "Leakage columns excluded from X",
    "PASS" if not any(c in X_train_reduced.columns for c in leakage_or_audit_cols) else "FAIL",
    f"Excluded columns: {leakage_or_audit_cols}"
)

add_qc(
    "Stratified train/test split performed",
    "PASS",
    f"Train n={len(y_train)}, test n={len(y_test)}, test_size={TEST_SIZE}, random_state={RANDOM_STATE}"
)

add_qc(
    "Constant predictors removed using training data only",
    "PASS",
    f"Dropped: {constant_cols}"
)

add_qc(
    "Preprocessor fit on training data only",
    "PASS",
    "fit_transform applied to training set; transform applied to test set."
)

add_qc(
    "No missing values after preprocessing",
    "PASS" if (X_train_processed_df.isna().sum().sum() == 0 and X_test_processed_df.isna().sum().sum() == 0) else "FAIL",
    f"Train missing={int(X_train_processed_df.isna().sum().sum())}; test missing={int(X_test_processed_df.isna().sum().sum())}"
)

add_qc(
    "No ML model training performed",
    "PASS",
    "This notebook only creates train/test splits and preprocessing outputs."
)

qc = pd.DataFrame(qc_items)
display(qc)
qc.to_csv(OUTPUT_DIR / "12_quality_control_checklist.csv", index=False)

if (qc["status"] == "FAIL").any():
    raise RuntimeError("One or more QC checks failed. Review 12_quality_control_checklist.csv before proceeding.")

In [ ]:
# ============================================================
# 13. Summary report
# ============================================================

summary = f"""
Notebook 04 — Preprocessing Pipeline and Train/Test Split
========================================================================

Input feature matrix:
{INPUT_FILE}

Input rows: {df.shape[0]}
Input candidate predictors before constant-column removal: {len(feature_cols)}

Target:
- Column: {target_col}
- Positive n: {int(y.sum())}
- Negative n: {int((1-y).sum())}
- Positive percent: {100*y.mean():.2f}%

Train/test split:
- Train n: {len(y_train)}
- Test n: {len(y_test)}
- Test size: {TEST_SIZE}
- Random state: {RANDOM_STATE}
- Stratified by target: yes

Constant predictors removed using training data only:
{constant_cols}

Feature types after constant-column removal:
- Continuous predictors: {len(continuous_features)}
- Binary predictors: {len(binary_features)}
- Categorical predictors: {len(categorical_features)}

Processed feature matrix:
- Train processed shape: {X_train_processed_df.shape}
- Test processed shape: {X_test_processed_df.shape}
- Missing values after preprocessing: train={int(X_train_processed_df.isna().sum().sum())}, test={int(X_test_processed_df.isna().sum().sum())}

Important methodological note:
- No imputation/scaling/encoding information from the test set was used to fit preprocessing.
- No machine learning model was trained in this notebook.
- Baseline_NP3TOT is retained in the primary predictor set but a sensitivity analysis excluding it is planned because the outcome is based on change in NP3TOT.

Generated files:
- 01_train_test_split_summary.csv
- 02_dropped_constant_predictors.csv
- 03_feature_type_dictionary.csv
- 04_train_raw_split_before_preprocessing.csv
- 05_test_raw_split_before_preprocessing.csv
- 06_train_processed_matrix.csv
- 07_test_processed_matrix.csv
- 08_processed_feature_names.csv
- 09_fitted_preprocessing_pipeline.joblib
- 10_sensitivity_feature_set_plan.csv
- 11_processed_feature_names_without_baseline_NP3TOT.csv
- 12_quality_control_checklist.csv
- 13_notebook_04_summary_report.txt

Output folder:
{OUTPUT_DIR}
"""

print(summary)

with open(OUTPUT_DIR / "13_notebook_04_summary_report.txt", "w", encoding="utf-8") as f:
    f.write(summary)

## Scientific Interpretation

If all QC checks pass, the data are ready for the first modeling notebook. The next stage should compare simple and interpretable baseline models before complex models.

Recommended next notebook:

**Notebook 05 — Baseline Machine Learning Models and Internal Validation**

This next stage should include:
- Logistic regression as a baseline interpretable model.
- Random forest and gradient boosting as non-linear comparators.
- Stratified cross-validation on the training set only.
- Final evaluation on the held-out test set.
- Metrics suitable for class imbalance: ROC-AUC, PR-AUC, balanced accuracy, sensitivity, specificity, F1-score, and calibration.